# Program 07: Feature Engineering Exploratory Data Analysis
**Student Name**: VEDANT NIMKAR  
**Registration Number**: 26MML0045  
**Course**: MACSE502 - Python for Data Science Lab  
**Week**: 04 | **Date**: 30-07-2026 | **Type**: PP  

## Problem Statement
Create a DataFrame with columns Feature1, Feature2, and Target containing random data. Perform the following operations: create a new feature that is the logarithm of Feature1, bin Feature2 into three categories (low, medium, high), and calculate the correlation matrix of the DataFrame. Finally, create a scatter plot of Feature1 vs. Feature2 colored by Target, and interpret any visible patterns.

## Objectives
- To generate a reproducible synthetic dataset using the NumPy random module.
- To understand feature engineering as the creation of new columns from existing ones.
- To apply a logarithmic transformation in order to compress a skewed feature.
- To convert a continuous variable into ordered categories using binning.
- To quantify the linear association between variables using a correlation matrix.
- To visualize two features against each other with the class label shown as colour, and to interpret the pattern that results.


In [ ]:
# --- Cell 1: Generated Dataset with Feature1, Feature2 and Target ---
# Load Necessary Packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Fix the random seed for reproducibility
np.random.seed(42)

n = 200

# Step 2: Generate the raw features
feature1 = np.random.lognormal(mean=3.0, sigma=0.6, size=n)   # right skewed, positive
feature2 = np.random.normal(loc=50, scale=15, size=n)         # roughly symmetric

# Derive the binary Target from a weighted score plus noise
score = 0.35 * feature1 + 0.90 * feature2 + np.random.normal(0, 8, n)
target = (score > np.median(score)).astype(int)

# Step 3: Assemble the DataFrame
df = pd.DataFrame({
    'Feature1': feature1.round(2),
    'Feature2': feature2.round(2),
    'Target': target
})

print("First 10 rows of the generated dataset")
print(df.head(10))
print("\nShape =", df.shape)
print("Target distribution:")
print(df['Target'].value_counts())


In [ ]:
# --- Cell 2: Descriptive Statistics and Skewness of the Raw Features ---
# Step 4: Inspect the raw features before engineering
print("Descriptive statistics of the raw dataset")
print(df.describe())

print("\nSkewness of Feature1 =", round(df['Feature1'].skew(), 4))
print("Skewness of Feature2 =", round(df['Feature2'].skew(), 4))


In [ ]:
# --- Cell 3: Engineered Log_Feature1 Column and its Effect on Skewness ---
# Step 5: Feature engineering - logarithm of Feature1
df['Log_Feature1'] = np.log(df['Feature1'])

print("Dataset after adding the engineered Log_Feature1 column")
print(df.head(10))

print("\nSkewness before log transform =", round(df['Feature1'].skew(), 4))
print("Skewness after  log transform =", round(df['Log_Feature1'].skew(), 4))
print("\nRange of Feature1     :", df['Feature1'].min(), "to", df['Feature1'].max())
print("Range of Log_Feature1 :", round(df['Log_Feature1'].min(), 4),
      "to", round(df['Log_Feature1'].max(), 4))


In [ ]:
# --- Cell 4: Feature2 Binned into Low, Medium and High Categories ---
# Step 6: Feature engineering - bin Feature2 into three categories
df['Feature2_Category'] = pd.cut(df['Feature2'],
                                 bins=3,
                                 labels=['low', 'medium', 'high'])

print("Dataset after binning Feature2 into three categories")
print(df.head(10))

print("\nNumber of observations in each category:")
print(df['Feature2_Category'].value_counts().sort_index())

print("\nFeature2 range covered by each bin:")
print(df.groupby('Feature2_Category', observed=True)['Feature2'].agg(['min', 'max', 'count']))


In [ ]:
# --- Cell 5: Correlation Matrix of the Numeric Columns ---
# Step 7: Correlation matrix of the numeric columns
numeric_cols = ['Feature1', 'Feature2', 'Log_Feature1', 'Target']
corr = df[numeric_cols].corr()

print("Correlation matrix")
print(corr.round(4))

print("\nCorrelation of each feature with the Target:")
print(corr['Target'].drop('Target').sort_values(ascending=False).round(4))


In [ ]:
# --- Cell 6: Scatter Plot of Feature1 vs Feature2 Coloured by Target ---
# Step 8: Scatter plot of Feature1 vs Feature2 coloured by Target
plt.figure(figsize=(8, 5.5))

for cls, colour, marker in [(0, '#1f77b4', 'o'), (1, '#d62728', '^')]:
    subset = df[df['Target'] == cls]
    plt.scatter(subset['Feature1'], subset['Feature2'],
                c=colour, marker=marker, s=45, alpha=0.75,
                edgecolors='white', linewidths=0.5,
                label=f'Target = {cls}')

plt.xlabel('Feature1')
plt.ylabel('Feature2')
plt.title('Feature1 vs Feature2 coloured by Target')
plt.legend(title='Class')
plt.grid(True, linestyle='--', alpha=0.35)
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 7: Group-wise Summary Confirming the Visual Pattern ---
# Step 9: Confirm the visual pattern numerically
summary = df.groupby('Feature2_Category', observed=True).agg(
    Records=('Target', 'size'),
    Mean_Feature1=('Feature1', 'mean'),
    Mean_Feature2=('Feature2', 'mean'),
    Target_Rate=('Target', 'mean')
).round(3)

print("Group-wise summary by Feature2 category")
print(summary)

print("\nMean Feature2 by class:")
print(df.groupby('Target')['Feature2'].mean().round(3))
print("\nMean Feature1 by class:")
print(df.groupby('Target')['Feature1'].mean().round(3))


## Conclusion
This experiment successfully demonstrated a complete feature engineering and exploratory data analysis workflow using Pandas, NumPy and Matplotlib together. A reproducible synthetic dataset of 200 observations was generated with a deliberate relationship built into it, two new features were engineered from the raw columns, and the structure of the data was then examined both numerically and graphically. The results showed that:

- The raw Feature1 was strongly right skewed, and the natural logarithm reduced that skewness towards zero while compressing a wide multiplicative range into a narrow one.
- pd.cut() with bins=3 produced equal width rather than equal frequency categories, which is why the medium band contained far more observations than the outer bands.
- Feature1 and Log_Feature1 were almost perfectly correlated, confirming that a monotonic transformation reshapes a distribution without adding new information.
- The correlation matrix identified Feature2 as the feature more strongly associated with the Target, consistent with the larger weight used when the Target was derived.
- The scatter plot revealed the two classes separating mainly along the Feature2 axis with substantial overlap along Feature1, and the group-wise Target rate rising from the low band to the high band confirmed that pattern numerically.

Overall, this exercise showed that feature engineering and exploratory analysis are complementary rather than separate activities: the skewness statistic motivated the log transform, the correlation matrix indicated which feature mattered, and the scatter plot then made the class structure visible in a way no single number could. Equally important was the practice of verifying each visual impression against an aggregate statistic. These techniques form the standard first stage of any Data Science or Machine Learning project, since the quality of the engineered features usually influences model performance more than the choice of algorithm itself.